# Étape 9 — Phase D : Inference Phase C

Pipeline source : `artifacts/pipeline_phaseC.pkl`

- 7 modèles : F1_effb0 / F2_convnext / M3_convnext_seed1 / S5_swin_tiny / D7_mae_vit_b / D8_dinov2_vit_b / D9_facenet
- Ensemble = blend (alpha=0.725) :
  - stacking_pred : LightGBM 5-fold (14 features = 7 preds + gender + p_mean/std/min/max/range/median)
  - weighted_pred  : sum(w_i * pred_i) avec poids OOF moyens
- TTA flip horizontal pour chaque modèle

## 1. Setup

In [1]:
import sys, time, pickle, gc
from pathlib import Path

ROOT = Path.cwd()   # racine du depot (les notebooks sont a la racine)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader

from dc_utils.paths import IMAGE_DIR, ARTIFACTS_DIR
from dc_utils.training import (
    OcclusionDatasetRAM, default_eval_tf, preload_images_decoded,
)
import dc_utils.training as _t

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device, '| torch', torch.__version__, '| timm', timm.__version__)

EXP_DIR = ROOT / 'experiments'
DATA_DIR = ROOT / 'data'
SUB_DIR  = ROOT / 'submissions'; SUB_DIR.mkdir(exist_ok=True)

# --- Load pipeline Phase C ---
with open(ARTIFACTS_DIR / 'pipeline_phaseC.pkl', 'rb') as f:
    pipeline = pickle.load(f)
print('phase :', pipeline['phase'])
print('blend_alpha :', pipeline['blend_alpha'])
print('models :', pipeline['model_names'])
print('feature_columns :', pipeline['stacking']['feature_columns'])
print('CV blend score :', pipeline['val_score_5foldCV_blend']['score'])

<repo>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device = cuda | torch 2.2.2+cu121 | timm 1.0.27
phase : C
blend_alpha : 0.7250000000000001
models : ['F1_effb0', 'F2_convnext', 'M3_convnext_seed1', 'S5_swin_tiny', 'D7_mae_vit_b', 'D8_dinov2_vit_b', 'D9_facenet']
feature_columns : ['pred_F1_effb0', 'pred_F2_convnext', 'pred_M3_convnext_seed1', 'pred_S5_swin_tiny', 'pred_D7_mae_vit_b', 'pred_D8_dinov2_vit_b', 'pred_D9_facenet', 'gender', 'p_mean', 'p_std', 'p_min', 'p_max', 'p_range', 'p_median']
CV blend score : 0.0009828760032691327


## 2. df_test + gender via identity-join

In [ ]:
df_test_full = pd.read_csv(DATA_DIR / 'test_clean.csv')
df_train     = pd.read_csv(DATA_DIR / 'train_clean.csv')
print('test_clean :', df_test_full.shape, '| needs_imputation :', df_test_full.needs_imputation.sum())

# --- gender via identity-join ---
gender_by_id = (df_train.groupby('identity')['gender']
                .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan))
fallback_gender = float(df_train['gender'].mean())
df_test_full['gender'] = df_test_full['identity'].map(gender_by_id).astype(float)
n_mapped = df_test_full['gender'].notna().sum()
df_test_full['gender'] = df_test_full['gender'].fillna(fallback_gender)
print(f'gender mapped via identity : {n_mapped}/{len(df_test_full)} '
      f'({n_mapped/len(df_test_full)*100:.1f}%) — fallback {fallback_gender:.4f} pour le reste')

df_test = df_test_full[~df_test_full.needs_imputation].reset_index(drop=True).copy()
df_imp  = df_test_full[ df_test_full.needs_imputation].reset_index(drop=True).copy()
print(f'df_test (avec image) : {len(df_test)} | df_imp (sans image, à imputer) : {len(df_imp)}')

# Sanity check 
_missing = [fn for fn in df_test['filename'].head(20) if not (IMAGE_DIR / fn).exists()]
assert not _missing, f'Missing files in IMAGE_DIR: {_missing[:3]}'
print('image dir OK')

test_clean : (29980, 4) | needs_imputation : 118
gender mapped via identity : 28036/29980 (93.5%) — fallback 0.6764 pour le reste
df_test (avec image) : 29862 | df_imp (sans image, à imputer) : 118
image dir OK


## 3. Définitions modèles + loaders ckpt

In [3]:
from facenet_pytorch import InceptionResnetV1

class TimmRegressor(nn.Module):
    def __init__(self, backbone_name: str, pretrained: bool = True, bias_init: float | None = None,
                 model_kwargs: dict | None = None):
        super().__init__()
        kw = model_kwargs or {}
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=1, **kw)
        if bias_init is not None:
            head = self.backbone.get_classifier()
            with torch.no_grad():
                head.bias.fill_(bias_init); head.weight.mul_(0.01)
    def forward(self, x):
        return torch.sigmoid(self.backbone(x)).squeeze(-1)

class FaceNetRegressor(nn.Module):
    _IM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    _IM_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    def __init__(self, pretrained: str | None = 'vggface2', bias_init: float | None = None):
        super().__init__()
        self.backbone = InceptionResnetV1(pretrained=pretrained)
        self.head = nn.Linear(512, 1)
        if bias_init is not None:
            with torch.no_grad():
                self.head.bias.fill_(bias_init); self.head.weight.mul_(0.01)
        self.register_buffer('im_mean', self._IM_MEAN.clone())
        self.register_buffer('im_std',  self._IM_STD.clone())
    def get_classifier(self): return self.head
    def forward(self, x):
        x = x * self.im_std + self.im_mean
        x = F.interpolate(x, size=(160, 160), mode='bilinear', align_corners=False)
        x = (x - 0.5) / 0.5
        emb = self.backbone(x)
        return torch.sigmoid(self.head(emb)).squeeze(-1)

def load_timm_ckpt(backbone_name: str, ckpt_path: Path, model_kwargs: dict | None = None) -> nn.Module:
    m = TimmRegressor(backbone_name, pretrained=False, model_kwargs=model_kwargs).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(state['model']); m.eval()
    val = state.get('val', {})
    print(f'  loaded {ckpt_path.name} | best ep={state.get("epoch")} | val={val.get("score", float("nan")):.5f}')
    return m

def load_facenet_ckpt(ckpt_path: Path) -> nn.Module:
    m = FaceNetRegressor(pretrained=None).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = state['model']
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in unexpected if not k.startswith('backbone.logits.')]
    assert not missing,  f'Missing keys: {missing}'
    assert not bad,      f'Unexpected (autres que logits): {bad}'
    m.eval()
    val = state.get('val', {})
    print(f'  loaded {ckpt_path.name} (FaceNet) | best ep={state.get("epoch")} | val={val.get("score", float("nan")):.5f}')
    return m

print('classes + loaders OK')

classes + loaders OK


## 4. Charge les 7 modèles depuis le pipeline

In [4]:
models = {}
for spec in pipeline['models']:
    name = spec['name']
    ckpt = Path(spec['ckpt_path'])
    if spec['kind'] == 'timm':
        models[name] = load_timm_ckpt(spec['backbone'], ckpt, model_kwargs=spec.get('kwargs') or None)
    elif spec['kind'] == 'facenet':
        models[name] = load_facenet_ckpt(ckpt)
    else:
        raise ValueError(f'unknown kind: {spec["kind"]}')
print(f'\n{len(models)} modèles chargés OK')
print(f'GPU : {torch.cuda.memory_allocated()/1e9:.2f} GB')

  loaded exp_F1_effb0.pt | best ep=7 | val=0.00120
  loaded exp_F2_convnext.pt | best ep=7 | val=0.00116
  loaded exp_M3_convnext_seed1.pt | best ep=6 | val=0.00116
  loaded exp_S5_swin_tiny.pt | best ep=6 | val=0.00110
  loaded exp_D7_mae_vit_b.pt | best ep=8 | val=0.00126
  loaded exp_D8_dinov2_vit_b.pt | best ep=17 | val=0.00122
  loaded exp_D9_facenet.pt (FaceNet) | best ep=9 | val=0.00123

7 modèles chargés OK
GPU : 1.14 GB


## 5. Précharge des images test en RAM

In [5]:
t0 = time.time()
preload_images_decoded(df_test['filename'].tolist(), IMAGE_DIR, img_size=224, n_threads=16)
print(f'preload done in {time.time()-t0:.1f}s | RAM cache : {_t._IMG_ARRAY.shape}, {_t._IMG_ARRAY.nbytes/1e9:.2f} GB')

decode 29862 imgs: 100%|██████████| 29862/29862 [01:05<00:00, 458.81it/s]

  -> RAM cache: shape (29862, 224, 224, 3) | dtype uint8 | 4.50 GB (single ndarray, fork-safe)
preload done in 65.8s | RAM cache : (29862, 224, 224, 3), 4.50 GB


## 6. Prédictions TTA pour les 7 modèles + checkpoint

In [ ]:
@torch.inference_mode()
def predict_tta(model, dl, device, use_amp: bool = True):
    model.eval()
    preds, fns = [], []
    ctx = torch.amp.autocast('cuda', enabled=use_amp)
    for batch in dl:
        X = batch[0].to(device, non_blocking=True)
        fn = batch[-1]
        with ctx:
            p = model(X).float()
            p_flip = model(torch.flip(X, dims=[-1])).float()
            p = 0.5 * (p + p_flip)
        preds.append(p.cpu().numpy()); fns.extend(fn)
    return np.concatenate(preds), fns

ds_test = OcclusionDatasetRAM(df_test, IMAGE_DIR, default_eval_tf(), has_target=False)
dl_test = DataLoader(ds_test, batch_size=192, shuffle=False, num_workers=4, pin_memory=True)

_, FILENAMES_TEST = predict_tta(next(iter(models.values())), dl_test, device)
assert FILENAMES_TEST == df_test['filename'].tolist(), 'order mismatch'

PREDS_TEST: dict[str, np.ndarray] = {}
for name, m in models.items():
    t0 = time.time()
    p, fns = predict_tta(m, dl_test, device)
    assert fns == FILENAMES_TEST
    PREDS_TEST[name] = p.astype(np.float32)
    print(f'{name:22s} TTA | mean {p.mean():.4f} | std {p.std():.4f} | {time.time()-t0:.0f}s')

# Checkpoint disque 
np.savez(EXP_DIR / 'phaseD_preds_test.npz',
         filenames=np.array(FILENAMES_TEST), **PREDS_TEST)
print(f'\nsaved phaseD_preds_test.npz ({len(PREDS_TEST)} models, {len(FILENAMES_TEST)} samples)')

# Libère GPU et RAM cache
del models
torch.cuda.empty_cache()
_t._IMG_ARRAY = None; _t._IMG_INDEX = {}
gc.collect()
print(f'GPU : {torch.cuda.memory_allocated()/1e9:.2f} GB | RAM cache freed')

F1_effb0               TTA | mean 0.1630 | std 0.1007 | 19s
F2_convnext            TTA | mean 0.1669 | std 0.1028 | 26s
M3_convnext_seed1      TTA | mean 0.1651 | std 0.0990 | 26s
S5_swin_tiny           TTA | mean 0.1646 | std 0.1005 | 35s
D7_mae_vit_b           TTA | mean 0.1645 | std 0.1021 | 45s
D8_dinov2_vit_b        TTA | mean 0.1639 | std 0.1007 | 63s
D9_facenet             TTA | mean 0.1635 | std 0.0996 | 22s

saved phaseD_preds_test.npz (7 models, 29862 samples)
GPU : 0.11 GB | RAM cache freed


## 7. Build features pour le stacking

In [ ]:
model_names = pipeline['model_names']
feature_columns = pipeline['stacking']['feature_columns']

P_test = np.stack([PREDS_TEST[n] for n in model_names]) 
gender_test = (df_test['gender'].values).astype(np.float32)

def build_features(P_arr, gender_arr, names):
    feats = {f'pred_{names[i]}': P_arr[i] for i in range(len(names))}
    feats['gender']   = gender_arr.astype(np.float32)
    feats['p_mean']   = P_arr.mean(axis=0)
    feats['p_std']    = P_arr.std(axis=0)
    feats['p_min']    = P_arr.min(axis=0)
    feats['p_max']    = P_arr.max(axis=0)
    feats['p_range']  = feats['p_max'] - feats['p_min']
    feats['p_median'] = np.median(P_arr, axis=0)
    return pd.DataFrame(feats)

X_test = build_features(P_test, gender_test, model_names)
# Reorder to match training feature order
X_test = X_test[feature_columns]
print('X_test :', X_test.shape, '| cols :', X_test.columns.tolist())
assert X_test.columns.tolist() == feature_columns, 'feature mismatch'

X_test : (29862, 14) | cols : ['pred_F1_effb0', 'pred_F2_convnext', 'pred_M3_convnext_seed1', 'pred_S5_swin_tiny', 'pred_D7_mae_vit_b', 'pred_D8_dinov2_vit_b', 'pred_D9_facenet', 'gender', 'p_mean', 'p_std', 'p_min', 'p_max', 'p_range', 'p_median']


## 8. Stacking 5-fold LGBM (moyenne) + weighted ensemble + blend

In [ ]:
import lightgbm as lgb

# --- stacking ---
stk_pred = np.zeros(len(df_test), dtype=np.float64)
for path in pipeline['stacking']['boosters_files']:
    booster = lgb.Booster(model_file=path)
    stk_pred += booster.predict(X_test, num_iteration=booster.best_iteration)
stk_pred /= len(pipeline['stacking']['boosters_files'])
stk_pred = np.clip(stk_pred, 0.0, 1.0).astype(np.float32)
print(f'stacking_pred  : mean {stk_pred.mean():.4f} | std {stk_pred.std():.4f}')

# --- weighted ensemble 
weights = np.array([next(m['weight'] for m in pipeline['models'] if m['name'] == n) for n in model_names],
                   dtype=np.float64)
weights = weights / weights.sum()  # safety
w_pred = (weights[:, None] * P_test).sum(axis=0)
w_pred = np.clip(w_pred, 0.0, 1.0).astype(np.float32)
print(f'weighted_pred  : mean {w_pred.mean():.4f} | std {w_pred.std():.4f}')
print('weights :', dict(zip(model_names, np.round(weights, 4))))

# --- blend ---
alpha = pipeline['blend_alpha']
blend = alpha * stk_pred + (1.0 - alpha) * w_pred
blend = np.clip(blend, 0.0, 1.0).astype(np.float32)
print(f'\nblend (alpha={alpha:.3f}) : mean {blend.mean():.4f} | std {blend.std():.4f} | '
      f'min {blend.min():.4f} | max {blend.max():.4f}')

stacking_pred  : mean 0.1656 | std 0.1009
weighted_pred  : mean 0.1642 | std 0.0996
weights : {'F1_effb0': 0.2712, 'F2_convnext': 0.0853, 'M3_convnext_seed1': 0.1569, 'S5_swin_tiny': 0.2188, 'D7_mae_vit_b': 0.0071, 'D8_dinov2_vit_b': 0.0531, 'D9_facenet': 0.2075}

blend (alpha=0.725) : mean 0.1652 | std 0.1005 | min 0.0065 | max 0.4246


## 9. Imputation des 118 lignes sans image

Valeur d'imputation = moyenne pondérée train (`w_i = 1/30 + GT_i`).

In [9]:
# Valeur imputée = E_w[FaceOcclusion] avec w = 1/30 + GT (cohérent avec la métrique)
_w = (1.0 / 30.0 + df_train['FaceOcclusion'].values)
IMPUT_VALUE = float(np.sum(_w * df_train['FaceOcclusion'].values) / np.sum(_w))
print(f'IMPUT_VALUE (moyenne pondérée train) = {IMPUT_VALUE:.5f}')

# Construction du dict filename -> pred
preds_full: dict[str, float] = dict(zip(df_test['filename'].tolist(), blend.astype(float).tolist()))
for fn in df_imp['filename'].tolist():
    preds_full[fn] = IMPUT_VALUE

assert len(preds_full) == len(df_test_full), 'pred count mismatch'
print(f'preds_full : {len(preds_full)} entries (dont {len(df_imp)} imputées)')

IMPUT_VALUE (moyenne pondérée train) = 0.14699
preds_full : 29980 entries (dont 118 imputées)


## 10. Save submission CSV

In [10]:
df_students = pd.read_csv(DATA_DIR / 'test_students.csv')
assert set(df_students['filename']) == set(preds_full.keys()), 'filename set mismatch'

sub = pd.DataFrame({
    'filename': df_students['filename'],
    'FaceOcclusion': df_students['filename'].map(preds_full),
    'gender': 'x',
})
assert sub['FaceOcclusion'].notna().all(), 'NaN in submission'
assert (sub['FaceOcclusion'] >= 0).all() and (sub['FaceOcclusion'] <= 1).all()

out_path = SUB_DIR / 'submission_phaseC_blend.csv'
sub.to_csv(out_path, index=False)
n_imp = (sub['FaceOcclusion'] == IMPUT_VALUE).sum()
print(f'Submission écrite : {out_path}')
print(f'  n = {len(sub)} | imputés = {n_imp} | mean = {sub.FaceOcclusion.mean():.4f} | std = {sub.FaceOcclusion.std():.4f}')
print(sub.head(3))

# Stats par db (sanity check)
db_stats = (df_test_full.merge(sub, on='filename')
            .groupby('db')['FaceOcclusion']
            .agg(['count', 'mean', 'std', 'min', 'max']))
print('\nStats par database :'); print(db_stats)

Submission écrite : <repo>/submissions/submission_phaseC_blend.csv
  n = 29980 | imputés = 118 | mean = 0.1651 | std = 0.1003
                                            filename  FaceOcclusion gender
0  database3/database3/m.0256vn/81-FaceId-0_align...       0.043722      x
1  database3/database3/m.01dgwg/29-FaceId-0_align...       0.030600      x
2  database3/database3/m.02l0nk/72-FaceId-0_align...       0.019251      x

Stats par database :
           count      mean       std       min       max
db                                                      
database1   1427  0.188188  0.099204  0.006735  0.418638
database2    118  0.146990  0.000000  0.146990  0.146990
database3  28435  0.164030  0.100421  0.006462  0.424590
